In [2]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-04-21 12:51:36--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 

185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.1’

input.txt.1         100%[===================>]   1.06M   644KB/s    in 1.7s    

2026-04-21 12:51:38 (644 KB/s) - ‘input.txt.1’ saved [1115394/1115394]



In [3]:
with open('input.txt', 'r',encoding = 'utf-8') as f:
    text = f.read()

In [4]:
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [5]:
# all the unique chars occuring int the corpus
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(vocab_size)

65


In [6]:
#tokenizing
# encoder and decoder just like makemore architecture

stoi = {ch:i for i , ch in enumerate(chars)}
itos = {i:ch for i, ch in enumerate(chars)}

#print(stoi)

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])
#print(encode("hello"))

In [7]:
import torch
data = torch.tensor(encode(text), dtype = torch.long)
print(data.shape, data.dtype)
print(data[:100])

/home/shrys/dev/nanogpt/.venv/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:307: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [8]:
n = int(0.9*(len(data)))
train_data = data[:n]
val_data = data[n:]

In [ ]:
torch.manual_seed(42)
batch_size = 4 # number of independent sequences
block_size = 8

def get_batch(split):
    # generates a small btach of data
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data)- block_size,(batch_size,)) # random offsets 
    x = torch.stack([data[i:i+block_size]for i in ix] )
    y = torch.stack([data[i+1:i+block_size+1] for i in ix] )
    x, y = x.to(device), y.to(device)
    return x,y 


# average out loss over multiple batches 

@torch.no_grad()
def estimate_loss():
    out ={}
    model.eval() # setting model to eval phase
    for split in ['train','val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X,Y = get_batch(split)
            logits, loss = model(X,Y)
            losses[k] = loss.item()

        out[split] = losses.mean()
    model.train() # setting model to training phase
    return out     

xb, yb = get_batch('train')

#print('inputs:')
#print(xb.shape)
#print('targets')
#print(yb.shape)
#print('---------')

for b in range(batch_size):
    for t in range(block_size):
        context = xb[b:t+1]
        target = yb[b:t]

In [ ]:
import torch 
import torch.nn as nn
from torch.nn import  functional as F

torch.manual_seed(42)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits=  self.token_embedding_table(idx) # batch, time, channel
        
        if targets is None:
            loss = None

        else:    
            # B -> batch size, T-> context length(number of tokens)
            B,T,C = logits.shape
            logits = logits.view(B*T, C) # torch expects logits to have C as the second parameter
            targets = targets.view(B*T) # same reason to change the view as above
            loss = F.cross_entropy(logits,targets)
        return logits, loss

# idx is some intial token sequence, for starting the prediction
    def generate(self, idx, max_new_tokens):
        # idx is (B,T) array of indices in the current context
        for _ in range(max_new_tokens):
            logits, loss = self(idx)# forward pass 
            logits = logits[:,-1,:]
            probs = F.softmax(logits, dim =1)
            idx_next = torch.multinomial(probs, num_samples = 1)
            idx = torch.cat((idx, idx_next), dim =1) # (B, T+1)

        return idx   

m = BigramLanguageModel(vocab_size)     
logits, loss = m(xb,yb)
print(logits.shape)   
print(loss)

print(decode(m.generate(idx = torch.zeros((1,1), dtype = torch.long),max_new_tokens = 100)[0].tolist()))

torch.Size([32, 65])
tensor(4.8865, grad_fn=<NllLossBackward0>)

o$,q&IWqW&xtCjaB?ij&bYRGkF?b; f ,CbwhtERCIfuWr,DzJERjhLlVaF&EjffPHDFcNoGIG'&$qXisWTkJPw
 ,b Xgx?D3sj


In [ ]:
optimizer = torch.optim.AdamW(m.parameters(), lr = 1e-3)
batch_size = 32

for iter in range(max_iters):

    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f'step {iter}: train loss {losses['train']: .4f}, val loss {losses['val']: .4f}')


    xb,yb = get_batch('train')


    # evaluate the loss    
    logits , loss =m(xb,yb)
    optimizer.zero_grad(set_to_none = True)
    loss.backward()
    optimizer.step()


#print(loss.item())

# generate from the model
context = torch.zeros((1,1), dtype = torch.long, device =device)
print(decode(m.generate(context,max_new_tokens = 500)[0].tolist()))

2.4818594455718994




be beabbacou, asl abusan:
OWabeld pu l sceangous t.
Front. m.
BOFou aurn ce,
NGo masthth'sufoy fobl acatoine a mpty BOR:
I RORINGoflfeecarll ho ate pacose thyathe t pr,
Ancen an acant withenvee'?
Paissto,
Fovid LK:
Was s a thed'd-d tope e, ag antow


d wathur y RKII pavenw ore, o ty ad ts, wotert de st?
Has fron: t crstou wa utheraghesat douesto arwany, busw, the hoty ent r!
DIThigs,
I tola rol thoucased.
S:
Th thany, t t s bo:

t mapalentenoret temere bjorstonetu ma corere y fator wa t he THe 
